# Clustering template — mC + 3C embedding & integration (one major type)

This is the **per-major-type template** used to build the joint methylation + chromatin-contact
embedding and the consensus Leiden clusters. It is run once per major type; shown here for one
example (**Mus-Skl**). Each figure notebook reads the resulting `5kCG100k3C_embed.h5ad` /
`5kCG_embed.h5ad` / `100k3C_embed.h5ad`; this template shows how those are generated.


## 📥 Inputs & upstream provenance

- **`5kCG_embed.h5ad`** (input) — per-cell **5-kb CG LSI** (`tf-idf`→`TruncatedSVD`), computed from the single-cell **mcds** by **ALLCools** (upstream `01.5kCG_clustering`).
- **`100k3C_embed.h5ad`** (input) — per-cell **100-kb contact PCA**, from the **cell×binpair npz** by **scHiCluster**.
- **`tissue/L2/*/mCG_5kb_seurat_anchor_*.hdf`**, **`HiC_100kb_seurat_anchor_*.hdf`** — cross-donor/cross-cell-type **Seurat integration anchors**, computed with **biosketch-based downsampling** (upstream anchor step).
- Integration uses `ALLCools.integration.SeuratIntegration`; clustering uses `ALLCools.clustering.ConsensusClustering` (repeated Leiden + consensus).

_Writes (cache-guarded by `repro_guard`): `5kCG_embed.h5ad`, `100k3C_embed.h5ad`, `5kCG100k3C_embed.h5ad`._


In [1]:
import os, sys
ENTEX_ROOT = os.environ.get('ENTEX_ROOT', '/large_storage/zhoulab/zhoujt/project/ENTEx')
BOOK_ROOT  = os.environ.get('BOOK_ROOT',  f'{ENTEX_ROOT}/analysis/HumanCellEpigenomeAtlas')
sys.path.insert(0, BOOK_ROOT); import repro_guard
group_name = 'Mus-Skl'   # example major type; the template is run once per major type
os.chdir(f'{ENTEX_ROOT}/clustering/merged/L2/{group_name}')


## 1 · mC embedding — Seurat-anchor integration of the 5-kb CG LSI across donors & cell types


In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
from glob import glob

import anndata
import scanpy as sc
import scanpy.external as sce
from sklearn.preprocessing import normalize

from ALLCools.clustering import *
from ALLCools.plot import *
from ALLCools.integration.seurat_class import SeuratIntegration

mpl.style.use('default')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = 'Helvetica'


In [3]:
def dump_embedding(adata, name, n_dim=2):
    # put manifold coordinates into adata.obs
    for i in range(n_dim):
        adata.obs[f'{name}_{i}'] = adata.obsm[f'X_{name}'][:, i]
    return adata


In [4]:
# Parameters
cpu = 8
group_name = "Mus-Skl"
mem_gb = 30


In [5]:
mcad = anndata.read_h5ad('5kCG_embed.h5ad')
mcad

In [6]:
npc = pd.read_csv(f'{ENTEX_ROOT}/clustering/merged/npc.tsv', sep='\t', header=None, index_col=0).loc[group_name, 1]
print(npc)

In [7]:
sample_list = np.sort(mcad.obs['Donor'].astype(str).unique())
sample_list

In [8]:
mcad.obs['Donor'].value_counts()

In [9]:
adata_list = [mcad[mcad.obs['Donor']==xx] for xx in sample_list]


In [10]:
anchor_index = {}
for xx,adata in zip(sample_list, adata_list):
    tmp = pd.DataFrame(index=adata.obs.index).reset_index().reset_index()
    anchor_index[xx] = tmp.set_index('cell')['index'].to_dict()


In [11]:
outdir = f'{ENTEX_ROOT}/clustering/merged/'
ct_list = np.sort([xx.split('/')[-2] for xx in glob(f'{outdir}tissue/L2/*-*/mCG_5kb_seurat_anchor_*.hdf')])
print(len(ct_list))


In [12]:
anchor = []
for ct in ct_list:
    anchor_file = glob(f'{outdir}tissue/L2/{ct}/mCG_5kb_seurat_anchor_*.hdf')[0]
    adata_file = glob(f'{outdir}tissue/L2/{ct}/mCG_5kb_lsi.h5ad')[0]
    anchor_tmp = pd.read_hdf(anchor_file)
    adata_tmp = anndata.read_h5ad(adata_file)
    sample_tmp = np.sort(adata_tmp.obs['Donor'].unique())
    cell_list = [adata_tmp.obs.index[adata_tmp.obs['Donor']==xx] for xx in sample_tmp]
    thres = np.percentile(anchor_tmp['dist_pc'], 90)
    anchor_tmp = anchor_tmp.loc[anchor_tmp['dist_pc']<thres, ['x1','x2','score']].copy()
    anchor_tmp['x1'] = cell_list[0][anchor_tmp['x1']]
    anchor_tmp['x2'] = cell_list[1][anchor_tmp['x2']]
    anchor_tmp = anchor_tmp.loc[anchor_tmp['x1'].isin(mcad.obs.index) & anchor_tmp['x2'].isin(mcad.obs.index)]
    if anchor_tmp.shape[0]>0:
        anchor_tmp['x1'] = anchor_tmp['x1'].map(anchor_index[sample_tmp[0]])
        anchor_tmp['x2'] = anchor_tmp['x2'].map(anchor_index[sample_tmp[1]])
        anchor_tmp['x1_donor'] = sample_tmp[0]
        anchor_tmp['x2_donor'] = sample_tmp[1]
        anchor.append(anchor_tmp)
    
anchor = pd.concat(anchor, axis=0)


In [13]:
anchor[['x1_donor','x2_donor']].value_counts().sort_index()


In [14]:
n = len(adata_list)
anchor_df = {}

for i in range(n - 1):
    for j in range(i + 1, n):
        x1, x2 = sample_list[[i,j]]
        selc = (anchor['x1_donor']==x1) & (anchor['x2_donor']==x2)
        anchor_df[(i,j)] = anchor.loc[selc, ['x1', 'x2', 'score']]
        if selc.sum()==0:
            selc = (anchor['x1_donor']==x2) & (anchor['x2_donor']==x1)
            anchor_df[(i,j)] = anchor.rename({'x1':'x2','x2':'x1'}, axis=1).loc[selc, ['x1', 'x2', 'score']]
        

In [15]:
integrator = SeuratIntegration()


In [16]:
integrator.adata_dict = {k: v for k, v in zip(list(range(len(adata_list))), adata_list)}
integrator.n_dataset = len(adata_list)
integrator.n_cells = [adata.shape[0] for adata in adata_list]

# intra-dataset KNN for scoring the anchors
integrator.k_local = None
integrator.key_local = 'X_lsi'
integrator._calculate_local_knn()

# alignments and all_pairs
integrator.alignments = None
integrator._get_all_pairs()
integrator.anchor = anchor_df


In [17]:
corrected = integrator.integrate(key_correct='5kCG_lsi',
                                 row_normalize=True,
                                 n_components=npc,
                                 k_weight=100,
                                 sd=1)
corrected = pd.DataFrame(normalize(np.concatenate(corrected, axis=0), axis=1), 
                         index=np.concatenate([xx.obs.index for xx in adata_list]))

mcad.obsm[f'5kCG_u{npc}_seuratL2'] = corrected.loc[mcad.obs.index].values
mcad.obsm[f'5kCG_u{npc}_seuratL2'] = normalize(mcad.obsm[f'5kCG_u{npc}_seuratL2'][:, :npc], axis=1)

tsne(mcad, obsm=f'5kCG_u{npc}_seuratL2', metric='euclidean', exaggeration=-1, perplexity=50, n_jobs=-1)
mcad.obsm[f'5kCG_u{npc}_seuratL2_tsne'] = mcad.obsm['X_tsne'].copy()


In [18]:
mcad.write_h5ad('5kCG_embed.h5ad')

## 2 · 3C embedding — Seurat-anchor integration of the 100-kb contact PCA


In [19]:
# Parameters
cpu = 8
group_name = "Mus-Skl"
mem_gb = 30


In [20]:
indir = f'{ENTEX_ROOT}/clustering/merged/'


In [21]:
chromsize = pd.read_csv('/ref/hg38/fasta/hg38.main.chrom.sizes', sep='\t', index_col=0, header=None)
chromsize = chromsize.iloc[:22]


In [22]:
mcad = anndata.read_h5ad('100k3C_embed.h5ad')
mcad

In [23]:
npc = pd.read_csv(f'{ENTEX_ROOT}/clustering/merged/npc.tsv', sep='\t', header=None, index_col=0).loc[group_name, 2]
print(npc)

In [24]:
sample_list = np.sort(mcad.obs['Donor'].astype(str).unique())
sample_list

In [25]:
mcad.obs['Donor'].value_counts()

In [26]:
adata_list = [mcad[mcad.obs['Donor']==xx] for xx in sample_list]


In [27]:
anchor_index = {}
for xx,adata in zip(sample_list, adata_list):
    tmp = pd.DataFrame(index=adata.obs.index).reset_index().reset_index()
    anchor_index[xx] = tmp.set_index('cell')['index'].to_dict()


In [28]:
ct_list = np.sort([xx.split('/')[-2] for xx in glob(f'{indir}tissue/L2/*-*/HiC_100kb_seurat_anchor_*.hdf')])
print(len(ct_list))


In [29]:
anchor = []
for ct in ct_list:
    anchor_file = glob(f'{indir}tissue/L2/{ct}/HiC_100kb_seurat_anchor_*.hdf')[0]
    adata_file = glob(f'{indir}tissue/L2/{ct}/HiC_100kb_pca.h5ad')[0]
    anchor_tmp = pd.read_hdf(anchor_file)
    adata_tmp = anndata.read_h5ad(adata_file)
    sample_tmp = np.sort(adata_tmp.obs['Donor'].unique())
    cell_list = [adata_tmp.obs.index[adata_tmp.obs['Donor']==xx] for xx in sample_tmp]
    thres = np.percentile(anchor_tmp['dist_pc'], 90)
    anchor_tmp = anchor_tmp.loc[anchor_tmp['dist_pc']<thres, ['x1','x2','score']].copy()
    anchor_tmp['x1'] = cell_list[0][anchor_tmp['x1']]
    anchor_tmp['x2'] = cell_list[1][anchor_tmp['x2']]
    anchor_tmp = anchor_tmp.loc[anchor_tmp['x1'].isin(mcad.obs.index) & anchor_tmp['x2'].isin(mcad.obs.index)]
    if anchor_tmp.shape[0]>0:
        anchor_tmp['x1'] = anchor_tmp['x1'].map(anchor_index[sample_tmp[0]])
        anchor_tmp['x2'] = anchor_tmp['x2'].map(anchor_index[sample_tmp[1]])
        anchor_tmp['x1_donor'] = sample_tmp[0]
        anchor_tmp['x2_donor'] = sample_tmp[1]
        anchor.append(anchor_tmp)
    
anchor = pd.concat(anchor, axis=0)


In [30]:
anchor[['x1_donor','x2_donor']].value_counts().sort_index()


In [31]:
n = len(adata_list)
anchor_df = {}

for i in range(n - 1):
    for j in range(i + 1, n):
        x1, x2 = sample_list[[i,j]]
        selc = (anchor['x1_donor']==x1) & (anchor['x2_donor']==x2)
        anchor_df[(i,j)] = anchor.loc[selc, ['x1', 'x2', 'score']]
        if selc.sum()==0:
            selc = (anchor['x1_donor']==x2) & (anchor['x2_donor']==x1)
            anchor_df[(i,j)] = anchor.rename({'x1':'x2','x2':'x1'}, axis=1).loc[selc, ['x1', 'x2', 'score']]
        

In [32]:
integrator = SeuratIntegration()


In [33]:
integrator.adata_dict = {k: v for k, v in zip(list(range(len(adata_list))), adata_list)}
integrator.n_dataset = len(adata_list)
integrator.n_cells = [adata.shape[0] for adata in adata_list]

# intra-dataset KNN for scoring the anchors
integrator.k_local = None
integrator.key_local = 'X_pca'
integrator._calculate_local_knn()

# alignments and all_pairs
integrator.alignments = None
integrator._get_all_pairs()
integrator.anchor = anchor_df


In [34]:
corrected = integrator.integrate(key_correct='100k3C_pca',
                                 row_normalize=True,
                                 n_components=npc,
                                 k_weight=100,
                                 sd=1)
corrected = pd.DataFrame(normalize(np.concatenate(corrected, axis=0), axis=1), 
                         index=np.concatenate([xx.obs.index for xx in adata_list]))

mcad.obsm[f'100k3C_pc{npc}_seuratL2'] = corrected.loc[mcad.obs.index].values
mcad.obsm[f'100k3C_pc{npc}_seuratL2'] = normalize(mcad.obsm[f'100k3C_pc{npc}_seuratL2'][:, :npc], axis=1)

tsne(mcad, obsm=f'100k3C_pc{npc}_seuratL2', metric='euclidean', exaggeration=-1, perplexity=50, n_jobs=8)
mcad.obsm[f'100k3C_pc{npc}_seuratL2_tsne'] = mcad.obsm['X_tsne'].copy()


In [35]:
mcad.write_h5ad('100k3C_embed.h5ad')


## 3 · Joint mC + 3C embedding


In [36]:
# Parameters
cpu = 8
group_name = "Mus-Skl"
mem_gb = 30


In [37]:
adata_mc = anndata.read_h5ad('5kCG_embed.h5ad')
adata_3c = anndata.read_h5ad('100k3C_embed.h5ad')
adata_3c = adata_3c[adata_mc.obs.index].copy()


In [38]:
# npc_cg = [int(xx.split('_')[1][1:]) for xx in adata_mc.obsm.keys() if '_seuratL2_tsne' in xx][0]
# # npc_3c = [int(xx.split('_')[1][1:]) for xx in adata_3c.obsm.keys() if '100k3C_u' in xx][0]
# npc_3c = [int(xx.split('_')[1][2:]) for xx in adata_3c.obsm.keys() if '_seuratL2_tsne' in xx][0]
npc_cg, npc_3c = pd.read_csv(f'{ENTEX_ROOT}/clustering/merged/npc.tsv', sep='\t', header=None, index_col=0).loc[group_name].values
print(npc_cg, npc_3c)


In [39]:
adata = anndata.AnnData(obs=adata_mc.obs)
# adata.obsm[f'5kCG100k3C_u{npc_cg}u{npc_3c}'] = np.hstack([normalize(adata_mc.obsm[f'5kCG_u{npc_cg}hm'], axis=1),
#                                                           normalize(adata_3c.obsm['X_pca'], axis=1)])
adata.obsm[f'5kCG100k3C_u{npc_cg}pc{npc_3c}'] = np.hstack([normalize(adata_mc.obsm[f'5kCG_u{npc_cg}_seuratL2'], axis=1),
                                                           normalize(adata_3c.obsm[f'100k3C_pc{npc_3c}_seuratL2'], axis=1)])
tsne(adata, obsm=f'5kCG100k3C_u{npc_cg}pc{npc_3c}', metric='euclidean', exaggeration=-1, perplexity=50, n_jobs=8)
adata.obsm[f'5kCG100k3C_u{npc_cg}pc{npc_3c}_tsne'] = adata.obsm['X_tsne'].copy()


In [40]:
# clustering name
clustering_name = 'L1'

# ConsensusClustering
# Important factores
n_neighbors = 25
leiden_resolution = 1.0
# this parameter is the final target that limit the total number of clusters
# Higher accuracy means more conservative clustering results and less number of clusters
target_accuracy = 0.9
min_cluster_size = 30

# Other ConsensusClustering parameters
metric = 'euclidean'
consensus_rate = 0.8
leiden_repeats = 500
random_state = 0
train_frac = 0.5
train_max_n = 500
max_iter = 50
n_jobs = 8

# Dendrogram via Multiscale Bootstrap Resampling
nboot = 10000
method_dist = 'correlation'
method_hclust = 'average'

plot_type = 'static'

In [41]:
cc = ConsensusClustering(model=None,
                         n_neighbors=n_neighbors,
                         metric=metric,
                         min_cluster_size=min_cluster_size,
                         leiden_repeats=leiden_repeats,
                         leiden_resolution=leiden_resolution,
                         consensus_rate=consensus_rate,
                         random_state=random_state,
                         train_frac=train_frac,
                         train_max_n=train_max_n,
                         max_iter=max_iter,
                         n_jobs=n_jobs,
                         target_accuracy=target_accuracy)


In [42]:
cc.fit_predict(adata.obsm[f'5kCG100k3C_u{npc_cg}pc{npc_3c}'])

In [43]:
adata.obs['leiden_cons'] = cc.label.copy()
adata.obs['leiden_cv'] = cc.cv_predicted_label.copy()


In [44]:
cc.save('ConcensusClustering.model.lib')
adata.write_h5ad('5kCG100k3C_embed.h5ad')


In [45]:
from sklearn.metrics import adjusted_rand_score as ARI
ARI(adata.obs['leiden_cons'], adata.obs['leiden_cv'])


In [46]:
if len(cc.step_data)>1:
    cc.target_accuracy = 0
    cc.supervise_learning()
    cc.final_evaluation()
    adata.obs['leiden_init'] = cc.label.copy()
else:
    adata.obs['leiden_init'] = adata.obs['leiden_cons'].copy()
    